# 📦 Датасет customer_support_tickets_200k
***История обращений в техподдержку крупной компании за 2022–2024 года***





In [69]:
from google.colab import files
uploaded = files.upload()

Saving customer_support_tickets_200k.csv to customer_support_tickets_200k (1).csv


In [70]:
# Загрузка данных
import pandas as pd
df = pd.read_csv('customer_support_tickets_200k.csv')

# Что загрузили
print(f'Записей: {f"{len(df):,}".replace(",", " ")} | Колонки: {len(df.columns)}')
print(f'Статусы: {df["status"].unique().tolist()}')

Записей: 200 000 | Колонки: 30
Статусы: ['Open', 'Closed', 'Pending Customer', 'In Progress', 'Resolved']


***Что видим?:***

  - 200 000 записей, 30 колонок
  - Статусы: `Open`, `Closed`, `Pending Customer`, `In Progress`, `Resolved`

**`Resolved`** и **`Closed`** означают, что тикет завершен, их берём для анализа


In [71]:
#Время решения завершенных тикетов (решенные и закрытые)
finished = df[df['status'].isin(['Resolved','Closed'])]['resolution_time_hours']

print(f'Завершено: {f"{len(finished):,}".replace(",", " ")} тикетов')
print(f'Среднее: {finished.mean():.2f} ч | Медиана: {finished.median():.2f} ч')


Завершено: 79 999 тикетов
Среднее: 120.63 ч | Медиана: 120.62 ч


  - Завершено 79 999 тикетов (~ 40%) из 200 000, остальные в работе
  - Среднее и медиана почти одинаковы (~ 120.6 часа, ~ 5 дней), распределение симметричное, нет сильных перекосов
  - <u><font color="red">**?:</font> 5 дней на решение это норма или стоит ускорить процесс решения**</u>

**Вывод:**
  >Время на решение обращения 5 дней допустимо для некритичных обращений, следует проверить распределение по **`priority`**, возможно срочные обращения обрабатываются быстрее

In [72]:
# Группируем завершенные тикеты по приоритету
priority_stats = df[df['status'].isin(['Resolved','Closed'])].groupby('priority')['resolution_time_hours'].agg([
    ('count', 'count'),
    ('median', 'median'),
    ('mean', 'mean')
]).round(2).sort_values('median')

# Показываем в виде таблицы
print(priority_stats.to_markdown())

| priority   |   count |   median |   mean |
|:-----------|--------:|---------:|-------:|
| Low        |   19884 |   120.43 | 120.74 |
| High       |   19994 |   120.64 | 120.19 |
| Medium     |   20020 |   120.64 | 120.74 |
| Urgent     |   20101 |   120.78 | 120.85 |


> 💡 **Вывод:**
> - Все приоритеты решаются за ~5 дней — разницы в скорости нет.
> - Гипотеза не подтвердилась: `Urgent` не решают быстрее.
> - **Возможные причины:**

>      • Приоритет назначается, но не влияет на маршрут тикета (единая очередь для всех приоритетов)
>      • Все обращения идут через единый SLA (5 дней)
>      • Поле `priority` заполняется постфактум или несогласованно
> - <font color="orange">**Что можно проверить?**</font>

>      • Влияет ли приоритет на удовлетворённость клиентов (`customer_satisfaction_score`)?
>      • Чаще ли эскалируют срочные тикеты (`escalated`)?
>      • Стоит ли пересмотреть процесс маршрутизации для `Urgent`?

In [73]:
df[['status', 'priority', 'customer_satisfaction_score', 'escalated']].head(10)

,status,priority,customer_satisfaction_score,escalated
0,Open,Urgent,5,No
1,Closed,Urgent,5,Yes
2,Closed,Medium,5,Yes
3,Closed,Medium,4,Yes
4,Pending Customer,High,5,Yes
5,In Progress,Low,5,No
6,Open,Medium,1,Yes
7,Resolved,Low,1,Yes
8,Open,Low,1,No
9,Closed,Urgent,1,Yes


In [74]:
# Только завершенные тикеты
finished = df[df['status'].isin(['Resolved', 'Closed'])]

# Групприровка по приоритету и подсчет медианы удовлетворенности и % эскалаций
priority_quality = finished.groupby('priority').agg({
    'customer_satisfaction_score': 'median',
    'escalated': lambda x: (x == 'Yes').mean() * 100
}).round(2)

# Переименовываем столбцы
priority_quality.columns = ['satisfaction_median', 'escalated_pct']

# Смотрим возможные значения приоритета
priority_list = finished['priority'].unique().tolist()
print(priority_list)

['Urgent', 'Medium', 'Low', 'High']


In [75]:
# Сортируем по приоритету (логичный порядок)
priority_order = ['Low', 'Medium', 'High', 'Urgent']
priority_quality = priority_quality.reindex(priority_order)

# # Показываем результат в виде таблицы
print(priority_quality.to_markdown())

| priority   |   satisfaction_median |   escalated_pct |
|:-----------|----------------------:|----------------:|
| Low        |                     3 |           49.94 |
| Medium     |                     3 |           50.33 |
| High       |                     3 |           49.58 |
| Urgent     |                     3 |           50.76 |


> 💡 **Вывод:**
> - Удовлетворённость: медиана = 3 для всех приоритетов одинаковая
> - Эскалации:  ~50% для всех приоритетов, `urgent` не эскалируют чаще
> - **Заключение:**
>> - Приоритет, по всей видимости, не влияет на процесс, не является рабочим инструментом
>> - Возможно, поле `priority` заполняется постфактум или не используется в маршрутизации

<font color = "purple">**Пример запроса от операционной команды:**</font>
> <u>*Почему в отчете за вчера выросло среднее время решения тикетов?*</u>

In [76]:
# Берем только завершенные тикеты и считаем медиану за неделю
dayly_median = df[df['status'].isin(['Resolved','Closed'])].groupby('ticket_created_date')['resolution_time_hours'].median().tail(7)

# Смотрим последние 7 дней
print(f'Медиана времени решения (в часах) за последние 7 дней:\n\n{dayly_median.to_markdown()}')

Медиана времени решения (в часах) за последние 7 дней:

| ticket_created_date   |   resolution_time_hours |
|:----------------------|------------------------:|
| 2024-12-25            |                 112.85  |
| 2024-12-26            |                 142.76  |
| 2024-12-27            |                 121.2   |
| 2024-12-28            |                 118.715 |
| 2024-12-29            |                 132.54  |
| 2024-12-30            |                 125     |
| 2024-12-31            |                 127.44  |


In [79]:
# Среднее за 7 дней принимаем за норму
norm = dayly_median.mean()
# Время решения за вреча
latest = dayly_median.iloc[-1]
# Считаем как изменилось время решения за последний день
change = (latest - norm) / norm * 100

print(f'Норма за последние 7 дней: {norm:.2f} ч')
print(f'Вчера: {latest:.2f} ч')
print(f'Отклонение: {change:.2f}')

Норма за последние 7 дней: 125.79 ч
Вчера: 127.44 ч
Отклонение: 1.31


><font color="purple">**Шаблон ответа для чата поддержки:**</font>
>
> Проверили данные за <u>***полный***</u> последний день:
> - Медиана времени решения: **127.44 ч** (норма: ~ **125.79 ч**)
> - Отклонение: **+ 1.31%** — в пределах статистической погрешности (**<5%**, условимся, что для операционных данных <5% будум считать статистическим шумом)
> - Возможные причины: незначительные колебания в рамках обычной волатильности
> - Рекомендация: мониторим в штатном режиме, дополнительных действий не требуется
